# Experiment Result Analysis

This notebook checks experiment result JSON files, writes CSV summaries, validates result-count consistency, and creates duration plots.

Run the cells from either the repository root or the `analysis/` directory. Generated files are written to `analysis/output/`.

In [1]:
from __future__ import annotations

import csv
import json
import math
import os
import re
from collections import defaultdict
from pathlib import Path
from statistics import mean, median
from typing import Any

try:
    from IPython.display import display
except ImportError:
    display = print

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "analysis" and (cwd.parent / "results").exists() else cwd
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = ROOT / "analysis" / "output"
PLOTS_DIR = OUTPUT_DIR / "plots"

RESULTS_DIR, OUTPUT_DIR

(PosixPath('/home/maarten/Documents/doctoraat/code/query-aggregator-evaluation/results'),
 PosixPath('/home/maarten/Documents/doctoraat/code/query-aggregator-evaluation/analysis/output'))

## Load Results

Each result JSON is normalized into one row with the core timing fields and metadata from `parameters`.

In [2]:
def infer_authorization_mode(file_name: str) -> str:
    match = re.search(r"-(no-auth|nondelegated|delegated)\.json$", file_name)
    return match.group(1) if match else ""


def infer_experiment_name(file_name: str) -> str:
    match = re.search(r"-(no-auth|nondelegated|delegated)\.json$", file_name)
    return file_name[: match.start()] if match else ""


def infer_iteration_args(experiment_id: str) -> str:
    marker = "_query-user"
    if marker not in experiment_id:
        return ""
    prefix = experiment_id.split(marker, 1)[0]
    return prefix.split("-", 1)[1] if "-" in prefix else prefix


def infer_execution_type(experiment_id: str) -> str:
    if "_aggregator_discovered" in experiment_id:
        return "aggregator-discovered"
    if "_aggregator" in experiment_id:
        return "aggregator"
    if experiment_id.endswith("_indexed-cache"):
        return "local-indexed-cache"
    return "local"


def infer_run(file_name: str) -> int:
    match = re.search(r"_run-(\d+)", file_name)
    return int(match.group(1)) if match else 0


def number(value: Any, default: float = 0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def load_results(results_dir: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for path in sorted(results_dir.glob("*.json")):
        with path.open("r", encoding="utf-8") as handle:
            data = json.load(handle)
        parameters = data.get("parameters") or {}
        file_name = path.name
        rows.append({
            "file": file_name,
            "experimentId": data.get("experimentId", ""),
            "totalDuration": number(data.get("totalDuration")),
            "dief100ms": number(data.get("dief100ms")),
            "dief1s": number(data.get("dief1s")),
            "dief10s": number(data.get("dief10s")),
            "totalResults": int(number(data.get("totalResults"))),
            "timestampCount": len(data.get("timestamps") or []),
            "experimentName": parameters.get("experimentName") or infer_experiment_name(file_name),
            "experimentType": parameters.get("experimentType", ""),
            "authorizationMode": parameters.get("authorizationMode") or infer_authorization_mode(file_name),
            "iterationName": parameters.get("iterationName", ""),
            "iterationArgs": parameters.get("iterationArgs") or infer_iteration_args(data.get("experimentId", "")),
            "executionType": parameters.get("executionType") or infer_execution_type(data.get("experimentId", "")),
            "cacheStrategy": parameters.get("cacheStrategy", ""),
            "measurementRun": int(number(parameters.get("measurementRun") or infer_run(file_name))),
            "totalHttpRequests": int(number(parameters.get("totalHttpRequests", parameters.get("totalHTTPRequests", 0)))),
            "resourceRequests": int(number(parameters.get("resourceRequests", 0))),
            "authorizationTokenRequests": int(number(parameters.get("authorizationTokenRequests", 0))),
            "numberOfTriples": int(number(parameters.get("numberOfTriples", 0))),
            "warmupRuns": int(number(parameters.get("warmupRuns", 0))),
            "recordedRuns": int(number(parameters.get("recordedRuns", 0))),
        })
    return rows


rows = load_results(RESULTS_DIR)
print(f"Loaded {len(rows)} result files from {RESULTS_DIR}")
if pd:
    display(pd.DataFrame(rows).head())
else:
    rows[:2]

Loaded 3945 result files from /home/maarten/Documents/doctoraat/code/query-aggregator-evaluation/results


## Validate Result Counts

The main correctness check compares `totalResults` across local, aggregator, and discovered aggregator variants for the same experiment, authorization mode, iteration, and recorded run.

In [3]:
def variant_key(row: dict[str, Any]) -> str:
    return f"{row['executionType']}|{row['cacheStrategy'] or '-'}"


def comparable_run_key(row: dict[str, Any]) -> tuple[Any, ...]:
    return (
        row["experimentName"],
        row["authorizationMode"],
        row["iterationName"],
        row["iterationArgs"],
        row["measurementRun"],
    )


def stable_variant_key(row: dict[str, Any]) -> tuple[Any, ...]:
    return (
        row["experimentName"],
        row["authorizationMode"],
        row["iterationName"],
        row["iterationArgs"],
        variant_key(row),
    )


def group_by(items: list[dict[str, Any]], key_fn):
    grouped = defaultdict(list)
    for item in items:
        grouped[key_fn(item)].append(item)
    return grouped


def validate(rows: list[dict[str, Any]]) -> dict[str, Any]:
    comparable_mismatches = []
    for key, group in group_by(rows, comparable_run_key).items():
        counts = defaultdict(list)
        for row in group:
            counts[row["totalResults"]].append({
                "file": row["file"],
                "variant": variant_key(row),
                "totalResults": row["totalResults"],
            })
        if len(counts) > 1:
            comparable_mismatches.append({
                "key": "|".join(map(str, key)),
                "counts": {str(count): values for count, values in counts.items()},
            })

    unstable_variants = []
    for key, group in group_by(rows, stable_variant_key).items():
        counts = sorted({row["totalResults"] for row in group})
        if len(counts) > 1:
            unstable_variants.append({
                "key": "|".join(map(str, key)),
                "counts": counts,
                "files": [
                    {
                        "file": row["file"],
                        "measurementRun": row["measurementRun"],
                        "totalResults": row["totalResults"],
                    }
                    for row in group
                ],
            })

    timestamp_mismatches = [
        {
            "file": row["file"],
            "totalResults": row["totalResults"],
            "timestampCount": row["timestampCount"],
        }
        for row in rows
        if row["timestampCount"] != row["totalResults"]
    ]

    ok = not comparable_mismatches and not unstable_variants and not timestamp_mismatches
    return {
        "ok": ok,
        "totals": {
            "files": len(rows),
            "comparableMismatches": len(comparable_mismatches),
            "unstableVariants": len(unstable_variants),
            "timestampMismatches": len(timestamp_mismatches),
        },
        "comparableMismatches": comparable_mismatches,
        "unstableVariants": unstable_variants,
        "timestampMismatches": timestamp_mismatches,
    }


validation = validate(rows)
validation["totals"], "OK" if validation["ok"] else "FAILED"

({'files': 3945,
  'comparableMismatches': 10,
  'unstableVariants': 0,
  'timestampMismatches': 0},
 'FAILED')

Show the first mismatches, if any.

In [4]:
validation["comparableMismatches"][:5]

[{'key': 'el-overview-complex-experiment|nondelegated|activities-complexity|complex_complex_20|1',
  'counts': {'20': [{'file': 'activities-complexity-complex_complex_20_query-user_complex_aggregator_discovered_run-1-nondelegated.json',
     'variant': 'aggregator-discovered|-',
     'totalResults': 20},
    {'file': 'activities-complexity-complex_complex_20_query-user_complex_file-cache_run-1-nondelegated.json',
     'variant': 'local|file-cache',
     'totalResults': 20},
    {'file': 'activities-complexity-complex_complex_20_query-user_complex_indexed-cache_run-1-nondelegated.json',
     'variant': 'local-indexed-cache|indexed-cache',
     'totalResults': 20},
    {'file': 'activities-complexity-complex_complex_20_query-user_complex_no-cache_run-1-nondelegated.json',
     'variant': 'local|no-cache',
     'totalResults': 20}],
   '16': [{'file': 'activities-complexity-complex_complex_20_query-user_complex_aggregator_run-1-nondelegated.json',
     'variant': 'aggregator|-',
     'tot

## Aggregate Timings

In [5]:
def round3(value: float) -> float:
    return round(value, 3)


def aggregate_key(row: dict[str, Any]) -> tuple[Any, ...]:
    return (
        row["experimentName"],
        row["authorizationMode"],
        row["iterationName"],
        row["iterationArgs"],
        row["executionType"],
        row["cacheStrategy"],
    )


def aggregate(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    aggregates = []
    for key, group in group_by(rows, aggregate_key).items():
        sample = group[0]
        durations = [row["totalDuration"] for row in group]
        result_counts = sorted({row["totalResults"] for row in group})
        aggregates.append({
            "experimentName": sample["experimentName"],
            "authorizationMode": sample["authorizationMode"],
            "iterationName": sample["iterationName"],
            "iterationArgs": sample["iterationArgs"],
            "executionType": sample["executionType"],
            "cacheStrategy": sample["cacheStrategy"],
            "runs": len(group),
            "totalResults": "|".join(map(str, result_counts)),
            "medianDurationMs": round3(median(durations)),
            "averageDurationMs": round3(mean(durations)),
            "minDurationMs": round3(min(durations)),
            "maxDurationMs": round3(max(durations)),
            "medianHttpRequests": round3(median(row["totalHttpRequests"] for row in group)),
            "medianResourceRequests": round3(median(row["resourceRequests"] for row in group)),
            "medianAuthorizationTokenRequests": round3(median(row["authorizationTokenRequests"] for row in group)),
        })
    return sorted(aggregates, key=lambda row: tuple(str(row[column]) for column in [
        "experimentName", "authorizationMode", "iterationArgs", "executionType", "cacheStrategy"
    ]))


aggregates = aggregate(rows)
if pd:
    display(pd.DataFrame(aggregates).head(20))
else:
    aggregates[:5]

## Write CSV and Validation Reports

In [6]:
SUMMARY_COLUMNS = [
    "file", "experimentName", "authorizationMode", "iterationName", "iterationArgs",
    "executionType", "cacheStrategy", "measurementRun", "totalResults", "timestampCount",
    "totalDuration", "dief100ms", "dief1s", "dief10s", "totalHttpRequests",
    "resourceRequests", "authorizationTokenRequests", "numberOfTriples",
]

AGGREGATE_COLUMNS = [
    "experimentName", "authorizationMode", "iterationName", "iterationArgs",
    "executionType", "cacheStrategy", "runs", "totalResults", "medianDurationMs",
    "averageDurationMs", "minDurationMs", "maxDurationMs", "medianHttpRequests",
    "medianResourceRequests", "medianAuthorizationTokenRequests",
]


def write_csv(path: Path, rows: list[dict[str, Any]], columns: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_validation_markdown(path: Path, validation: dict[str, Any]) -> None:
    lines = [
        "# Validation Report",
        "",
        f"Status: {'OK' if validation['ok'] else 'FAILED'}",
        "",
        f"Files checked: {validation['totals']['files']}",
        f"Comparable result-count mismatches: {validation['totals']['comparableMismatches']}",
        f"Unstable variant result counts: {validation['totals']['unstableVariants']}",
        f"Timestamp/result-count mismatches: {validation['totals']['timestampMismatches']}",
        "",
    ]
    sections = [
        ("Comparable Result-Count Mismatches", validation["comparableMismatches"]),
        ("Unstable Variant Result Counts", validation["unstableVariants"]),
        ("Timestamp/Result-Count Mismatches", validation["timestampMismatches"]),
    ]
    for title, issues in sections:
        lines += [f"## {title}", ""]
        if not issues:
            lines += ["None.", ""]
            continue
        for issue in issues[:50]:
            lines += ["```json", json.dumps(issue, indent=2), "```"]
        if len(issues) > 50:
            lines.append(f"Omitted {len(issues) - 50} additional issues. See validation.json.")
        lines.append("")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

write_csv(OUTPUT_DIR / "summary.csv", rows, SUMMARY_COLUMNS)
write_csv(OUTPUT_DIR / "aggregates.csv", aggregates, AGGREGATE_COLUMNS)
(OUTPUT_DIR / "validation.json").write_text(json.dumps(validation, indent=2) + "\n", encoding="utf-8")
write_validation_markdown(OUTPUT_DIR / "validation.md", validation)

print(f"Wrote CSV and validation reports to {OUTPUT_DIR}")

Wrote CSV and validation reports to /home/maarten/Documents/doctoraat/code/query-aggregator-evaluation/analysis/output


## Plot Median Duration

If `matplotlib` is available, this creates one SVG per experiment and authorization mode.

In [9]:
def variant_label(row: dict[str, Any]) -> str:
    execution = row["executionType"].replace("aggregator-discovered", "disc").replace("aggregator", "agg")
    cache = f"/{row['cacheStrategy'].replace('-cache', '')}" if row["cacheStrategy"] else ""
    return f"{row['iterationArgs']} {execution}{cache}"


def color_for(row: dict[str, Any]) -> str:
    if row["executionType"] == "aggregator-discovered":
        return "#2f7ed8"
    if row["executionType"] == "aggregator":
        return "#0f9960"
    if row["cacheStrategy"] == "indexed-cache":
        return "#8b5cf6"
    if row["cacheStrategy"] == "file-cache":
        return "#d97706"
    return "#6b7280"


def plot_duration_groups(aggregates: list[dict[str, Any]]) -> list[Path]:
    if plt is None:
        print("matplotlib is not installed; skipping plots.")
        return []

    written = []
    groups = group_by(aggregates, lambda row: (row["experimentName"], row["authorizationMode"]))
    for (experiment_name, authorization_mode), group in groups.items():
        group = sorted(group, key=lambda row: (row["iterationArgs"], variant_label(row)))
        labels = [variant_label(row) for row in group]
        values = [row["medianDurationMs"] for row in group]
        colors = [color_for(row) for row in group]

        width = max(10, len(group) * 0.35)
        fig, ax = plt.subplots(figsize=(width, 6))
        ax.bar(range(len(group)), values, color=colors)
        ax.set_title(f"{experiment_name} ({authorization_mode})")
        ax.set_ylabel("median duration (ms)")
        ax.set_xticks(range(len(group)))
        ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
        ax.grid(axis="y", color="#dddddd")
        fig.tight_layout()

        safe_name = re.sub(r"[^a-zA-Z0-9_.-]+", "_", f"{experiment_name}_{authorization_mode}")
        output_path = PLOTS_DIR / f"{safe_name}_duration.svg"
        fig.savefig(output_path)
        plt.close(fig)
        written.append(output_path)
    return written


plot_paths = plot_duration_groups(aggregates)
print(f"Wrote {len(plot_paths)} plot(s) to {PLOTS_DIR}")
plot_paths[:5]

matplotlib is not installed; skipping plots.
Wrote 0 plot(s) to /home/maarten/Documents/doctoraat/code/query-aggregator-evaluation/analysis/output/plots


[]

## Quick Views

In [8]:
if pd:
    aggregates_df = pd.DataFrame(aggregates)
    display(aggregates_df.sort_values(["experimentName", "authorizationMode", "iterationArgs", "executionType"]).head(50))
else:
    aggregates[:10]